## 1. Configuración

Es lo único que hay que editar.

# Validador de conjunto de datos — MCDI500

**Programación para la Ciencia de Datos · Magíster en Ciencia de Datos e Inteligencia Artificial**

---

Comprueba si el conjunto de datos que eligió su grupo cumple los requisitos del proyecto,
y genera un informe para adjuntar al entregable de la Fase 1.

**Cómo se usa:** complete la celda de configuración y ejecute el cuaderno completo con
*Kernel → Restart Kernel and Run All Cells*. No hay nada que programar.

> **Cuándo ejecutarlo.** Después de que ustedes hayan clasificado las variables en su
> diccionario. Esta herramienta contrasta esa clasificación, no la reemplaza: la última
> sección del cuaderno vuelve sobre esto.

In [1]:
from pathlib import Path
import pandas as pd

# Encontrar la raíz del proyecto desde Jupyter.
ubicacion = Path.cwd().resolve()

PROYECTO = next(
    (
        carpeta
        for carpeta in [ubicacion, *ubicacion.parents]
        if (carpeta / "data" / "raw" / "ens2016.xlsx").is_file()
    ),
    None,
)

if PROYECTO is None:
    raise FileNotFoundError(
        "No se encontró data/raw/ens2016.xlsx. "
        "Abra Jupyter desde la carpeta del proyecto."
    )

origen = PROYECTO / "data" / "raw" / "ens2016.xlsx"
carpeta_salida = PROYECTO / "data" / "processed"
carpeta_salida.mkdir(parents=True, exist_ok=True)

# Variables seleccionadas y ponderador combinado F1–F2.
VARIABLES = [
    "IdEncuesta",
    "FechaInicioF1",
    "Edad",
    "Sexo",
    "Zona",
    "HTA",
    "di3",
    "dis2",
    "IMC",
    "anos_estudio_MINSAL_1",
    "GPAQ",
    "as27",
    "as28",
    "Fexp_F1F2p_Corr",
    "Conglomerado",
    "Estrato",
]

print("Leyendo la base original...")
base = pd.read_excel(origen, sheet_name="Sheet1")

faltantes = [v for v in VARIABLES if v not in base.columns]

if faltantes:
    raise ValueError(
        f"No se encontraron estas variables: {faltantes}"
    )

seleccion = base[VARIABLES].copy()

archivo_salida = (
    carpeta_salida / "ens_variables_seleccionadas.xlsx"
)

seleccion.to_excel(archivo_salida, index=False)

print(f"Base original: {base.shape[0]} filas y {base.shape[1]} columnas")
print(f"Archivo seleccionado: {seleccion.shape[0]} filas y "
      f"{seleccion.shape[1]} columnas")
print("En este paso no se excluyeron personas ni se cambiaron códigos.")
print(f"Archivo guardado en: {archivo_salida}")

Leyendo la base original...
Base original: 6233 filas y 1173 columnas
Archivo seleccionado: 6233 filas y 16 columnas
En este paso no se excluyeron personas ni se cambiaron códigos.
Archivo guardado en: /Users/felipekaysdiaztoro/proyecto-grupo4-mcdi500/data/processed/ens_variables_seleccionadas.xlsx


In [4]:
# Identificar personas con ponderador F1-F2 valido.
peso = pd.to_numeric(
    seleccion["Fexp_F1F2p_Corr"], errors="coerce"
)

factor_valido = (
    peso.notna()
    & peso.gt(0)
    & peso.lt(float("inf"))
)

seleccion_f1f2 = seleccion.loc[factor_valido].copy()
seleccion_f1f2["Fexp_F1F2p_Corr"] = peso.loc[factor_valido]

archivo_f1f2 = (
    carpeta_salida / "ens_variables_f1f2.xlsx"
)

seleccion_f1f2.to_excel(archivo_f1f2, index=False)

resumen = pd.DataFrame({
    "indicador": [
        "Personas antes del filtro",
        "Personas con ponderador valido",
        "Personas excluidas por ponderador no valido",
        "Suma de ponderadores del subconjunto",
    ],
    "valor": [
        len(seleccion),
        len(seleccion_f1f2),
        len(seleccion) - len(seleccion_f1f2),
        seleccion_f1f2["Fexp_F1F2p_Corr"].sum(),
    ],
})

carpeta_docs = PROYECTO / "docs"
carpeta_docs.mkdir(parents=True, exist_ok=True)

resumen.to_csv(
    carpeta_docs / "resumen_filtro_f1f2.csv",
    index=False,
)

display(resumen)
print(f"Archivo generado: {archivo_f1f2}")

,indicador,valor
0,Personas antes del filtro,6.233000e+03
1,Personas con ponderador valido,5.520000e+03
2,Personas excluidas por ponderador no valido,7.130000e+02
3,Suma de ponderadores del subconjunto,1.451897e+07


Archivo generado: /Users/felipekaysdiaztoro/proyecto-grupo4-mcdi500/data/processed/ens_variables_f1f2.xlsx


## 2. Umbrales del curso

Los requisitos del proyecto, en un solo lugar.

In [5]:

RUTA = str(
    PROYECTO / "data" / "processed" / "ens_variables_f1f2.xlsx"
)

SEP = None
ENCODING = "utf-8"

carpeta_docs = PROYECTO / "docs"
carpeta_docs.mkdir(parents=True, exist_ok=True)

INFORME = str(
    carpeta_docs / "informe_validacion_f1f2.md"
)

In [6]:
from pathlib import Path
import pandas as pd

MIN_FILAS, MIN_COLUMNAS = 2000, 12
MIN_ROLES = 3                       # cuántos roles analíticos distintos debe combinar
NULOS_MIN, NULOS_MAX = 1.0, 60.0    # porcentajes
MAX_MB = 100                        # límite práctico de GitHub

# Requisitos que DESCALIFICAN el conjunto. Todo lo demás se informa como AVISO:
# son recomendaciones que enriquecen el trabajo, pero no lo impiden.
OBLIGATORIOS = {"filas", "columnas", "multivariado", "numerica", "categorica"}

## 3. Clasificación por rol analítico

El rol de una variable no es su tipo de dato: tres columnas `int64` pueden ser un
identificador, una discreta y una binaria, y cada una exige un preprocesamiento distinto.

Las reglas se aplican en cascada: fecha, binaria, numérica, y al final texto.

In [7]:
def clasificar(s: pd.Series) -> str:
    """Devuelve el rol analítico de una variable."""
    s = s.dropna()
    if s.empty:
        return "vacía"

    unicos = s.nunique()
    proporcion_unicos = unicos / len(s)
    es_numerica = pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s)

    if pd.api.types.is_datetime64_any_dtype(s):
        return "fecha"

    if unicos == 2:                                   # dos valores: sí/no, 0/1
        return "binaria"

    if es_numerica:
        es_entera = s.mod(1).eq(0).all()
        if es_entera and proporcion_unicos > 0.95 and len(s) > 50:
            return "identificador"
        if es_entera and (unicos <= 20 or proporcion_unicos < 0.05):
            return "discreta"
        return "continua"

    # --- texto ---
    if pd.to_datetime(s.head(200), errors="coerce", format="mixed").notna().mean() > 0.9:
        return "fecha"
    if s.astype(str).str.len().mean() > 60:
        return "texto libre"
    if proporcion_unicos > 0.95 and len(s) > 50:
        return "identificador"
    return "nominal" if unicos <= 15 else "alta cardinalidad"


assert clasificar(pd.Series([0, 1, 1, 0])) == "binaria"
assert clasificar(pd.Series([1.5, 2.7, 3.1] * 40)) == "continua"
assert clasificar(pd.Series(["Norte", "Sur", "Centro"] * 40)) == "nominal"
print("[OK] Casos de prueba superados")

[OK] Casos de prueba superados


## 4. Carga del archivo

Si el archivo no está donde indica `RUTA`, el cuaderno avisa y sigue: corrija la ruta y
vuelva a ejecutar.

In [8]:
ruta = Path(RUTA)
df = None

if not ruta.exists():
    print(f"[PENDIENTE] No se encontró el archivo: {ruta}")
    print("Corrija RUTA en la celda de configuración. La ruta se escribe desde")
    print("la carpeta donde está este cuaderno.")
else:
    try:
        if ruta.suffix.lower() in {".xlsx", ".xls"}:
            df = pd.read_excel(ruta)
        else:
            df = pd.read_csv(ruta, sep=SEP, encoding=ENCODING,
                             engine="python" if SEP is None else "c")
        print(f"[OK] {df.shape[0]} filas x {df.shape[1]} columnas")
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        print('Pruebe con SEP = ";" o ENCODING = "latin-1".')

[OK] 5520 filas x 16 columnas


## 5. Perfil de las variables

Una fila por variable, con su rol, sus valores distintos y su porcentaje de faltantes.

In [9]:
if df is not None:
    perfil = pd.DataFrame({
        "variable": df.columns,
        "dtype": [str(t) for t in df.dtypes],
        "rol": [clasificar(df[c]) for c in df.columns],
        "unicos": [df[c].nunique() for c in df.columns],
        "pct_nulos": (df.isna().mean() * 100).round(1).values,
    })
    display(perfil)
    print(perfil["rol"].value_counts().to_string())
else:
    perfil = None

,variable,dtype,rol,unicos,pct_nulos
0,IdEncuesta,int64,identificador,5520,0.0
1,FechaInicioF1,datetime64[us],fecha,5512,0.0
2,Edad,int64,discreta,83,0.0
3,Sexo,int64,binaria,2,0.0
4,Zona,int64,binaria,2,0.0
5,HTA,float64,binaria,2,0.2
6,di3,int64,discreta,3,0.0
7,dis2,int64,discreta,4,0.0
8,IMC,float64,continua,5332,0.7
9,anos_estudio_MINSAL_1,float64,discreta,23,0.9


rol
discreta         7
continua         4
binaria          3
identificador    1
fecha            1


## 6. Requisitos del curso

La condición de fondo es que el conjunto sea **multivariado**: que combine variables de
distinto tipo, porque cada tipo obliga a un tratamiento distinto y es ese tratamiento lo
que se evalúa.

**FALTA** descalifica el conjunto. **AVISO** no lo descalifica: señala algo que conviene
revisar o que enriquecería el trabajo si estuviera presente.

In [10]:
if perfil is not None:
    roles = perfil["rol"].value_counts()
    n = lambda rol: int(roles.get(rol, 0))

    # El identificador no cuenta como rol analítico: se excluye del análisis.
    roles_utiles = [r for r in roles.index if r not in ("identificador", "vacía")]
    numericas = n("continua") + n("discreta")
    categoricas = n("nominal") + n("binaria") + n("ordinal")
    alta = n("alta cardinalidad") + n("texto libre")

    filas, columnas = df.shape
    max_nulos = perfil["pct_nulos"].max()
    peso_mb = ruta.stat().st_size / 1024**2
    duplicadas = int(df.duplicated().sum())

    # (clave, requisito, se cumple, detalle)
    requisitos = [
        ("filas", f"Al menos {MIN_FILAS} filas", filas >= MIN_FILAS, f"{filas} filas"),
        ("columnas", f"Al menos {MIN_COLUMNAS} columnas", columnas >= MIN_COLUMNAS,
         f"{columnas} columnas"),
        ("multivariado", f"Combina al menos {MIN_ROLES} roles analíticos",
         len(roles_utiles) >= MIN_ROLES, ", ".join(roles_utiles)),
        ("numerica", "Al menos una variable numérica", numericas >= 1,
         f"{numericas} numéricas (continuas o discretas)"),
        ("categorica", "Al menos una variable categórica", categoricas >= 1,
         f"{categoricas} categóricas (nominales, binarias u ordinales)"),
        ("faltantes", "Presencia de valores faltantes", max_nulos >= NULOS_MIN,
         f"máximo {max_nulos}% en una variable"),
        ("nulos_extremos", f"Ninguna variable sobre {NULOS_MAX:.0f}% de faltantes",
         max_nulos <= NULOS_MAX, f"máximo {max_nulos}%"),
        ("fecha", "Incluye alguna variable de fecha", n("fecha") >= 1,
         f"{n('fecha')} detectadas"),
        ("alta_cardinalidad", "Incluye texto o categórica de alta cardinalidad",
         alta >= 1, f"{alta} detectadas"),
        ("tamano", f"Archivo bajo {MAX_MB} MB", peso_mb <= MAX_MB, f"{peso_mb:.1f} MB"),
        ("duplicados", "Sin filas duplicadas exactas", duplicadas == 0,
         "ninguna" if duplicadas == 0 else f"{duplicadas} duplicadas"),
    ]

    chequeo = pd.DataFrame(
        [{"estado": "OK" if ok else ("FALTA" if clave in OBLIGATORIOS else "AVISO"),
          "requisito": req, "detalle": det}
         for clave, req, ok, det in requisitos])
    display(chequeo)

    apto = "FALTA" not in chequeo["estado"].values
    avisos = int((chequeo["estado"] == "AVISO").sum())
    print("RESULTADO:", "el conjunto CUMPLE los requisitos mínimos."
          if apto else "el conjunto NO cumple. Revise lo marcado como FALTA.")
    if apto and avisos:
        print(f"Hay {avisos} aviso(s): no impiden usar el conjunto, pero conviene leerlos.")
else:
    chequeo, apto = None, False

,estado,requisito,detalle
0,OK,Al menos 2000 filas,5520 filas
1,OK,Al menos 12 columnas,16 columnas
2,OK,Combina al menos 3 roles analíticos,"discreta, continua, binaria, fecha"
3,OK,Al menos una variable numérica,11 numéricas (continuas o discretas)
4,OK,Al menos una variable categórica,"3 categóricas (nominales, binarias u ordinales)"
5,OK,Presencia de valores faltantes,máximo 18.0% en una variable
6,OK,Ninguna variable sobre 60% de faltantes,máximo 18.0%
7,OK,Incluye alguna variable de fecha,1 detectadas
8,AVISO,Incluye texto o categórica de alta cardinalidad,0 detectadas
9,OK,Archivo bajo 100 MB,0.5 MB


RESULTADO: el conjunto CUMPLE los requisitos mínimos.
Hay 1 aviso(s): no impiden usar el conjunto, pero conviene leerlos.


## 7. Informe para el entregable

In [11]:
def a_markdown(tabla: pd.DataFrame) -> str:
    """Convierte un DataFrame en una tabla Markdown."""
    encabezado = "| " + " | ".join(tabla.columns) + " |"
    separador = "| " + " | ".join("---" for _ in tabla.columns) + " |"
    filas = ["| " + " | ".join(str(v) for v in fila) + " |" for fila in tabla.values]
    return "\n".join([encabezado, separador, *filas])


if perfil is not None and INFORME:
    texto = "\n\n".join([
        "# Validación del dataset — MCDI500",
        f"**Archivo:** `{ruta.name}` · {df.shape[0]} filas × {df.shape[1]} columnas",
        "## Perfil de variables", a_markdown(perfil),
        "## Requisitos del curso", a_markdown(chequeo),
        "## Resultado",
        "El conjunto **cumple** los requisitos mínimos." if apto
        else "El conjunto **no cumple todavía** los requisitos mínimos.",
    ])
    Path(INFORME).write_text(texto, encoding="utf-8")
    print(f"Informe escrito en: {INFORME} — adjúntelo al entregable de la Fase 1.")

Informe escrito en: /Users/felipekaysdiaztoro/proyecto-grupo4-mcdi500/docs/informe_validacion_f1f2.md — adjúntelo al entregable de la Fase 1.


## 8. Contraste con la clasificación del equipo

**El validador puede equivocarse.** Clasifica con reglas generales, sin conocer el
significado de sus variables: una columna entera y casi única puede ser un identificador o
un folio con información, y una variable de dos valores puede ser binaria o una nominal de
dos categorías. Eso lo decide el equipo, no la herramienta.

Escriban abajo el rol que ustedes asignaron a cada variable. Cada discrepancia es una
decisión que deben justificar en el informe.

In [12]:
# {"nombre_columna": "rol"} — roles: continua, discreta, binaria, nominal, ordinal,
# fecha, alta cardinalidad, texto libre, identificador
CLASIFICACION_EQUIPO = {
    # "id": "identificador",
    # "edad": "continua",
}

if perfil is not None and CLASIFICACION_EQUIPO:
    comparacion = perfil[["variable", "rol"]].rename(columns={"rol": "segun_validador"})
    comparacion["segun_equipo"] = comparacion["variable"].map(CLASIFICACION_EQUIPO)
    comparacion = comparacion.dropna(subset=["segun_equipo"])
    comparacion["coincide"] = comparacion["segun_equipo"] == comparacion["segun_validador"]

    display(comparacion)
    discrepancias = comparacion.loc[~comparacion["coincide"], "variable"].tolist()
    print(f"Discrepancias: {len(discrepancias)}")
    for v in discrepancias:
        print("  · revisar y justificar:", v)
else:
    print("Complete CLASIFICACION_EQUIPO con el rol que ustedes asignaron.")

Complete CLASIFICACION_EQUIPO con el rol que ustedes asignaron.


---

*Validador de conjunto de datos · MCDI500 · Magíster en Ciencia de Datos e Inteligencia
Artificial · Universidad Andrés Bello*

In [13]:
from pathlib import Path

print(Path("informe_dataset.md").resolve())

/Users/felipekaysdiaztoro/proyecto-grupo4-mcdi500/F2/informe_dataset.md
